In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""Stacking 앙상블 백테스트
Walk-Forward 체크포인트를 사용하여 시간순 Stacking 백테스트.

전략:
  1. 각 3개월 윈도우의 TFT+LightGBM 체크포인트 로드
  2. 이전 윈도우 예측으로 메타모델(Stacking) 학습
  3. 현재 윈도우 예측으로 매수/매도 신호 생성
  4. Top-N 포트폴리오 시뮬레이션
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
RAW_MACRO_PATH = BASE_PATH / "raw" / "macro"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_backtest"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# 체크포인트 경로
TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

# 백테스트 기간: 2021 Q2 부터 (Q1은 메타모델 학습용)
BT_START_WINDOW = 1   # Window 1 (2021 Q2)부터 백테스트
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# 포트폴리오 설정
INITIAL_CAPITAL = 100_000_000
BUY_COMMISSION = 0.00015
SELL_COMMISSION = 0.00215
TOP_N = 10
STOP_LOSS = -0.03
TAKE_PROFIT = 0.07
MODEL_EXIT = 0.45
MAX_POSITIONS = 20

# ===== 보유기간 유연화 설정 =====
BASE_HOLD_DAYS = 5            # 기본 보유일 (기존 MAX_HOLD_DAYS)
HOLD_EXTEND_THRESHOLD = 0.52  # 이 확률 이상이면 보유 연장
MAX_HOLD_DAYS = 15            # 절대 최대 보유일 (연장 포함)
EXTENDED_STOP_LOSS = -0.02    # 연장 구간 타이트 손절선
EXTENDED_TAKE_PROFIT = 0.10   # 연장 구간 익절선 (추세 따라감)

# Walk-Forward 윈도우
WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3

windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"윈도우: {len(windows)}개, 백테스트: Window {BT_START_WINDOW}~{len(windows)-1}")
print(f"전략: Top-{TOP_N}, 손절 {STOP_LOSS*100}%, 익절 +{TAKE_PROFIT*100}%")
print(f"보유: 기본 {BASE_HOLD_DAYS}일 → 신호 유지 시 최대 {MAX_HOLD_DAYS}일 (연장 임계값={HOLD_EXTEND_THRESHOLD})")
print(f"연장 구간: 손절 {EXTENDED_STOP_LOSS*100}%, 익절 +{EXTENDED_TAKE_PROFIT*100}%")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): return self.loss_fn(self(b[0]),b[1])
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 정의 완료")

In [ ]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

In [ ]:
# ===== Step 1: 모든 윈도우의 예측 수집 =====
# 각 윈도우별로 TFT+LightGBM 예측을 모아서 하나의 DataFrame 생성

all_predictions = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    if not tft_ckpt.exists() or not lgbm_ckpt.exists():
        print(f"  [{i:2d}] SKIP: 체크포인트 없음")
        continue
    
    print(f"  [{i:2d}] {test_start}~{test_end} 예측 생성...")
    
    # TFT 예측
    tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
    tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    if len(tft_samples) < 10:
        del tft_model; torch.cuda.empty_cache(); continue
    tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
    del tft_model; torch.cuda.empty_cache()
    
    # LightGBM 예측
    lgbm_model = joblib.load(str(lgbm_ckpt))
    lgbm_probs, _, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
    del lgbm_model
    
    # GARCH
    garch_df = pd.read_parquet(str(garch_ckpt)) if garch_ckpt.exists() else None
    
    # 매칭
    tft_df = pd.DataFrame(tft_metas); tft_df["tft_prob"] = tft_probs; tft_df["label"] = tft_labels
    lgbm_df = pd.DataFrame(lgbm_metas); lgbm_df["lgbm_prob"] = lgbm_probs
    merged = tft_df.merge(lgbm_df, on=["date","ticker"], how="inner")
    merged["window"] = i
    
    if garch_df is not None and "ticker" in garch_df.columns:
        merged = merged.merge(garch_df[["ticker","vol_5d","risk_flag"]], on="ticker", how="left")
        merged["vol_5d"] = merged["vol_5d"].fillna(40.0)
        merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
    else:
        merged["vol_5d"] = 40.0; merged["risk_flag"] = 0
    
    all_predictions.append(merged)
    print(f"       {len(merged):,} 예측")

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f"\n전체 예측: {len(pred_df):,} rows, 윈도우 {pred_df['window'].nunique()}개")

In [ ]:
# ===== Step 2: 롤링 Stacking 메타모델로 앙상블 확률 생성 =====
# 각 윈도우 i에서: 이전 윈도우들(0~i-1)의 예측으로 메타모델 학습 → 윈도우 i 예측

pred_df["ensemble_prob"] = np.nan

for i in range(BT_START_WINDOW, pred_df["window"].max() + 1):
    # 이전 윈도우들로 메타모델 학습
    train_mask = pred_df["window"] < i
    test_mask = pred_df["window"] == i
    
    if train_mask.sum() < 100 or test_mask.sum() == 0:
        continue
    
    X_train = pred_df[train_mask][["tft_prob", "lgbm_prob"]].values
    y_train = pred_df[train_mask]["label"].values
    X_test = pred_df[test_mask][["tft_prob", "lgbm_prob"]].values
    
    meta = LogisticRegression(random_state=42, class_weight="balanced")
    meta.fit(X_train, y_train)
    
    ensemble_probs = meta.predict_proba(X_test)[:, 1]
    pred_df.loc[test_mask, "ensemble_prob"] = ensemble_probs
    
    da = accuracy_score(pred_df[test_mask]["label"], (ensemble_probs >= 0.5).astype(int))
    print(f"  Window [{i:2d}] meta학습={train_mask.sum():,} → 예측={test_mask.sum():,} DA={da*100:.1f}% "
          f"(TFT w={meta.coef_[0][0]:.2f}, LGBM w={meta.coef_[0][1]:.2f})")

# 앙상블 확률이 없는 행 제거
bt_df = pred_df.dropna(subset=["ensemble_prob"]).copy()
print(f"\n백테스트 대상: {len(bt_df):,} rows ({bt_df['date'].min()} ~ {bt_df['date'].max()})")

In [ ]:
# ===== Step 3: 날짜별 예측 dict 생성 =====
predictions = {}
for _, row in bt_df.iterrows():
    d, t = row["date"], row["ticker"]
    if d not in predictions: predictions[d] = {}
    predictions[d][t] = float(row["ensemble_prob"])

print(f"예측 거래일: {len(predictions)}, 기간: {min(predictions.keys())} ~ {max(predictions.keys())}")

In [ ]:
# ===== Step 4: 종가 데이터 로드 =====
tickers_needed = set()
for dp in predictions.values(): tickers_needed.update(dp.keys())

close_prices = {}
for ticker in tickers_needed:
    path = RAW_OHLCV_PATH / f"{ticker}.parquet"
    if not path.exists(): continue
    try:
        ohlcv = pd.read_parquet(str(path))
        ohlcv.index = pd.to_datetime(ohlcv.index)
        close_prices[ticker] = ohlcv["close"].sort_index()
    except: pass

kospi = pd.Series(dtype=float)
kospi_path = RAW_MACRO_PATH / "kospi200.parquet"
if kospi_path.exists():
    kdf = pd.read_parquet(str(kospi_path))
    kdf.index = pd.to_datetime(kdf.index)
    col = "close" if "close" in kdf.columns else kdf.columns[0]
    kospi = kdf[col].sort_index()

print(f"종가: {len(close_prices)} 종목, KOSPI200: {len(kospi)} rows")

In [ ]:
# ===== Step 5: 포트폴리오 시뮬레이션 (보유기간 유연화) =====
@dataclass
class Position:
    ticker: str; buy_date: str; buy_price: float; shares: int
    buy_prob: float; invested: float; hold_days: int = 0
    extended: bool = False   # 연장 구간 진입 여부
    peak_price: float = 0.0  # 연장 구간 트레일링용

def get_close(ticker, date):
    if ticker not in close_prices: return None
    s = close_prices[ticker]; ts = pd.Timestamp(date)
    if ts in s.index: return float(s.loc[ts])
    mask = s.index <= ts
    return float(s.loc[mask].iloc[-1]) if mask.any() else None

cash = INITIAL_CAPITAL
positions = []
trade_log = []
snapshots = []
extend_stats = {"extended": 0, "timeout_base": 0, "timeout_max": 0}

trading_dates = sorted(predictions.keys())
print(f"시뮬레이션: {len(trading_dates)} 거래일")
print(f"보유기간 유연화: 기본 {BASE_HOLD_DAYS}일 → 신호>{HOLD_EXTEND_THRESHOLD} 시 최대 {MAX_HOLD_DAYS}일\n")

for i, date in enumerate(trading_dates):
    for p in positions:
        p.hold_days += 1
    
    # 매도 체크
    date_preds = predictions.get(date, {})
    sells = []
    for pos in positions:
        price = get_close(pos.ticker, date)
        if price is None: continue
        ret = (price - pos.buy_price) / pos.buy_price
        
        # 연장 구간이면 peak_price 갱신
        if pos.extended:
            pos.peak_price = max(pos.peak_price, price)
        
        # --- 우선순위 1: 손절 (연장 구간은 타이트 손절) ---
        sl = EXTENDED_STOP_LOSS if pos.extended else STOP_LOSS
        if ret <= sl:
            tag = "ext_stop_loss" if pos.extended else "stop_loss"
            sells.append((pos, f"{tag} ({ret*100:.1f}%)")); continue
        
        # --- 우선순위 2: 모델 이탈 ---
        cp = date_preds.get(pos.ticker)
        if cp is not None and cp < MODEL_EXIT:
            sells.append((pos, f"model_exit (p={cp:.3f})")); continue
        
        # --- 우선순위 3: 익절 (연장 구간은 더 넓은 익절선) ---
        tp = EXTENDED_TAKE_PROFIT if pos.extended else TAKE_PROFIT
        if ret >= tp:
            tag = "ext_take_profit" if pos.extended else "take_profit"
            sells.append((pos, f"{tag} ({ret*100:.1f}%)")); continue
        
        # --- 우선순위 4: 보유기간 만료 판단 ---
        if pos.hold_days >= MAX_HOLD_DAYS:
            # 절대 상한 도달 → 무조건 매도
            extend_stats["timeout_max"] += 1
            sells.append((pos, f"timeout_max ({pos.hold_days}d)")); continue
        
        if pos.hold_days >= BASE_HOLD_DAYS and not pos.extended:
            # 기본 보유일 도달 → 신호 확인
            if cp is not None and cp >= HOLD_EXTEND_THRESHOLD and ret > 0:
                # 신호가 살아있고 수익 중 → 연장
                pos.extended = True
                pos.peak_price = price
                extend_stats["extended"] += 1
            else:
                # 신호 약하거나 손실 중 → 기존대로 매도
                extend_stats["timeout_base"] += 1
                sells.append((pos, f"timeout ({pos.hold_days}d)")); continue
        
        # 연장 구간 중 신호 재확인 (매일)
        if pos.extended and pos.hold_days > BASE_HOLD_DAYS:
            if cp is not None and cp < HOLD_EXTEND_THRESHOLD:
                sells.append((pos, f"ext_signal_fade (p={cp:.3f}, {pos.hold_days}d)")); continue
    
    for pos, reason in sells:
        price = get_close(pos.ticker, date)
        if price is None: continue
        rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
        cash += net; positions.remove(pos)
        trade_log.append({"date":date,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":net-pos.invested,
            "return":(net-pos.invested)/pos.invested,"hold_days":pos.hold_days,
            "reason":reason,"extended":pos.extended})
    
    # 매수
    preds = predictions[date]
    held = {p.ticker for p in positions}
    candidates = [(t,p) for t,p in preds.items() if t not in held and p > 0.5]
    candidates.sort(key=lambda x: -x[1])
    buys = candidates[:TOP_N]
    
    for ticker, prob in buys:
        if len(positions) >= MAX_POSITIONS: break
        price = get_close(ticker, date)
        if price is None or price <= 0: continue
        slots = MAX_POSITIONS - len(positions)
        alloc = min(cash / max(slots, 1), cash)
        shares = int(alloc / (price * (1 + BUY_COMMISSION)))
        if shares <= 0: continue
        cost = shares * price; comm = cost * BUY_COMMISSION; total = cost + comm
        if total > cash: continue
        cash -= total
        positions.append(Position(ticker, date, price, shares, prob, total, peak_price=price))
        trade_log.append({"date":date,"ticker":ticker,"action":"BUY","price":price,
            "shares":shares,"amount":total,"commission":comm,"prob":prob})
    
    pv = cash + sum(p.shares * (get_close(p.ticker, date) or 0) for p in positions)
    snapshots.append({"date":date,"portfolio_value":pv,"cash":cash,"n_positions":len(positions)})
    
    if (i+1) % 100 == 0:
        print(f"  Day {i+1}/{len(trading_dates)} | {date} | 자산: {pv:,.0f} | 수익: {(pv/INITIAL_CAPITAL-1)*100:+.2f}%")

# 잔여 청산
if positions:
    last = trading_dates[-1]
    print(f"\n잔여 {len(positions)} 포지션 청산")
    for pos in list(positions):
        price = get_close(pos.ticker, last)
        if price is None: continue
        rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
        cash += net; positions.remove(pos)
        trade_log.append({"date":last,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":net-pos.invested,
            "return":(net-pos.invested)/pos.invested,"hold_days":pos.hold_days,
            "reason":"backtest_end","extended":pos.extended})

print(f"\n시뮬레이션 완료")
print(f"보유 연장 통계: 연장={extend_stats['extended']}건, "
      f"기본만료={extend_stats['timeout_base']}건, 최대만료={extend_stats['timeout_max']}건")

In [ ]:
# ===== Step 6: 결과 계산 =====
snap_df = pd.DataFrame(snapshots)
snap_df["date"] = pd.to_datetime(snap_df["date"])
trade_df = pd.DataFrame(trade_log)
sell_df = trade_df[trade_df["action"]=="SELL"] if len(trade_df) > 0 else pd.DataFrame()

final_value = snap_df["portfolio_value"].iloc[-1]
total_return = final_value / INITIAL_CAPITAL - 1
n_days = len(snap_df)
annual_return = (1 + total_return) ** (252 / max(n_days, 1)) - 1

daily_ret = snap_df["portfolio_value"].pct_change().dropna()
sharpe = float(daily_ret.mean() / daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 1e-9 else 0
cummax = snap_df["portfolio_value"].cummax()
mdd = float(((snap_df["portfolio_value"] - cummax) / cummax).min())

if len(sell_df) > 0:
    win_rate = (sell_df["pnl"] > 0).mean()
    total_comm = trade_df["commission"].sum()
    avg_win = sell_df[sell_df["pnl"]>0]["return"].mean() if (sell_df["pnl"]>0).any() else 0
    avg_loss = sell_df[sell_df["pnl"]<=0]["return"].mean() if (sell_df["pnl"]<=0).any() else 0
    avg_hold = sell_df["hold_days"].mean()
else:
    win_rate = total_comm = avg_win = avg_loss = avg_hold = 0

# 벤치마크
bm_ret = None
bt_s = pd.Timestamp(snap_df["date"].iloc[0])
bt_e = snap_df["date"].iloc[-1]
bm_slice = kospi[(kospi.index >= bt_s) & (kospi.index <= bt_e)]
if len(bm_slice) >= 2:
    bm_ret = float(bm_slice.iloc[-1] / bm_slice.iloc[0] - 1)

print("="*70)
print("  Stacking 앙상블 백테스트 (보유기간 유연화)")
print("="*70)
print(f"  기간: {snap_df['date'].iloc[0].date()} ~ {snap_df['date'].iloc[-1].date()} ({n_days}일)")
print(f"  초기 자본: {INITIAL_CAPITAL:>15,}원")
print(f"  최종 자산: {final_value:>15,.0f}원")
print(f"  총 수익률: {total_return*100:>+8.2f}%")
print(f"  연환산:    {annual_return*100:>+8.2f}%")
print(f"  Sharpe:    {sharpe:>8.3f}")
print(f"  MDD:       {mdd*100:>8.2f}%")
if bm_ret is not None:
    print(f"  KOSPI200:  {bm_ret*100:>+8.2f}%")
    print(f"  초과수익:  {(total_return-bm_ret)*100:>+8.2f}%")
print(f"  총 거래:   {len(sell_df):>8}건")
print(f"  승률:      {win_rate*100:>8.1f}%")
print(f"  평균이익:  {avg_win*100:>+8.2f}%")
print(f"  평균손실:  {avg_loss*100:>+8.2f}%")
print(f"  평균보유:  {avg_hold:>8.1f}일")
print(f"  총수수료:  {total_comm:>12,.0f}원")
print()

if len(sell_df) > 0:
    reasons = sell_df["reason"].apply(lambda x: x.split(" ")[0]).value_counts()
    print("  [매도 사유]")
    for r, c in reasons.items():
        print(f"    {r:20s}: {c:5d}건 ({c/len(sell_df)*100:.1f}%)")

    # 연장 vs 비연장 성과 비교
    if "extended" in sell_df.columns:
        ext_sells = sell_df[sell_df["extended"] == True]
        noext_sells = sell_df[sell_df["extended"] == False]
        print(f"\n  [보유 연장 효과]")
        if len(noext_sells) > 0:
            print(f"    비연장: {len(noext_sells)}건, 승률={((noext_sells['pnl']>0).mean())*100:.1f}%, "
                  f"평균수익={noext_sells['return'].mean()*100:+.2f}%, 평균보유={noext_sells['hold_days'].mean():.1f}일")
        if len(ext_sells) > 0:
            print(f"    연장:   {len(ext_sells)}건, 승률={((ext_sells['pnl']>0).mean())*100:.1f}%, "
                  f"평균수익={ext_sells['return'].mean()*100:+.2f}%, 평균보유={ext_sells['hold_days'].mean():.1f}일")

print()
print("="*70)
print(f"\n  기존 (고정 5일):")
print(f"    수익률=+47.97%, Sharpe=0.528, MDD=-18.42%, 거래=5,299건, 수수료=73,017,581원")
print(f"\n비교:")
print(f"  LSTM Top-10:     수익률=+48.95%, Sharpe=1.974, MDD=-17.75%")
print(f"  TFT v2 Top-10:   수익률=+34.92%, Sharpe=1.567, MDD=-12.10%")

In [ ]:
# ===== 저장 =====
snap_df.to_parquet(str(MODEL_SAVE_PATH / f"equity_curve_flex_{datetime.now().strftime('%Y%m%d')}.parquet"))
if len(trade_df) > 0:
    trade_df.to_parquet(str(MODEL_SAVE_PATH / f"trade_log_flex_{datetime.now().strftime('%Y%m%d')}.parquet"))

summary = {"model": "Stacking_Ensemble_FlexHold",
  "period": f"{snap_df['date'].iloc[0].date()}~{snap_df['date'].iloc[-1].date()}",
  "total_return": round(total_return*100, 2), "annual_return": round(annual_return*100, 2),
  "sharpe": round(sharpe, 3), "mdd": round(mdd*100, 2),
  "win_rate": round(win_rate*100, 1), "total_trades": len(sell_df),
  "avg_hold_days": round(avg_hold, 1),
  "total_commission": round(total_comm),
  "benchmark": round(bm_ret*100, 2) if bm_ret else None,
  "flex_hold_params": {"base_hold": BASE_HOLD_DAYS, "max_hold": MAX_HOLD_DAYS,
    "extend_threshold": HOLD_EXTEND_THRESHOLD, "ext_stop_loss": EXTENDED_STOP_LOSS,
    "ext_take_profit": EXTENDED_TAKE_PROFIT},
  "extend_stats": extend_stats}
json.dump(summary, open(str(MODEL_SAVE_PATH/"backtest_summary_flex.json"),"w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## Stacking 앙상블 백테스트 (보유기간 유연화)

### 핵심 변경
```
기존: hold_days >= 5 → 무조건 매도 (timeout 33.4%)
개선: hold_days >= 5 → 신호 확인:
  - ensemble_prob >= 0.52 AND 수익 중 → 보유 연장 (최대 15일)
  - 연장 구간: 타이트 손절(-2%), 넓은 익절(+10%)
  - 연장 중 신호 < 0.52 → 즉시 매도 (ext_signal_fade)
  - 15일 도달 → 무조건 매도 (timeout_max)
```

### 기대 효과
1. timeout 매도 감소 → 거래 수 감소 → **수수료 절감** (기존 7,300만원)
2. 추세 따라가기 → 승리 트레이드의 수익 확대 (기존 평균 +4.46%)
3. 연장 구간 타이트 손절 → 반전 시 빠른 탈출

### 비교 (기존 vs 유연화)
| 지표 | 고정 5일 | 유연화 (5→15일) |
|------|---------|----------------|
| 수익률 | +47.97% | ? |
| Sharpe | 0.528 | ? |
| MDD | -18.42% | ? |
| 거래 수 | 5,299건 | ? |
| 수수료 | 73,017,581원 | ? |